In [225]:
import os
from ensurepip import bootstrap
from gc import set_debug, callbacks
from statistics import mean
from jupyter_builder.jupyterlab_semver import satisfies
from prompt_toolkit.key_binding.bindings.named_commands import yank_nth_arg
from sklearn.impute import SimpleImputer
from optuna.trial import FixedTrial
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from statsmodels.tools.sm_exceptions import iteration_limit_doc
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from sklearn.inspection import permutation_importance
import shap
from optuna.visualization import plot_param_importances, plot_optimization_history, plot_slice
import math
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import TargetEncoder
from sklearn.base import clone
import warnings

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="adult")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM adults", engine)
df['workclass'] = df['workclass'].fillna('Private')
df['native_country'] = df['native_country'].fillna('United-States')
df = df.drop_duplicates()
top_6 = ['Prof-specialty',
    'Craft-repair',
    'Exec-managerial',
    'Adm-clerical',
    'Sales',
    'Other-service']
prods = df['occupation'].value_counts().loc[top_6]
prods = prods / prods.sum()
null_mask = df['occupation'].isna()
rng = np.random.default_rng(42)
df.loc[null_mask, 'occupation'] = rng.choice(top_6, size=null_mask.sum(), p=prods)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
df.shape

(32537, 15)

In [226]:
x = df.drop(columns=['income',  'sex', 'education'])
y = df['income']

In [227]:
x_tmp, x_test, y_tmp, y_test  = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_tr, x_val, y_tr, y_val = train_test_split(x_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

In [228]:
t = ['workclass', 'marital_status', 'occupation', 'native_country', 'race', 'relationship']
prep = ColumnTransformer([('cat', TargetEncoder(cv=5, shuffle=True, random_state=42, smooth='auto'), t)], remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')

In [229]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_tr, y_tr)
print(accuracy_score(y_test, dummy.predict(x_test)))
print(roc_auc_score(y_test, dummy.predict_proba(x_test)[:, 1]))

0.7590657652120467
0.5


In [230]:
def unic(trial, model_name):
    if model_name == 'lr':
        params = {'C': trial.suggest_float('lr_C', 0.01, 2.0, log=True),
                  'penalty': trial.suggest_categorical('lr_penalty', ['l1', 'l2']),
                  'class_weight': trial.suggest_categorical('lr_class_weight', [None, 'balanced']),
                  'solver': 'liblinear',
                  'max_iter': 1000,
                  'random_state': 42}
        model = Pipeline([('scaler', StandardScaler()),
                          ('clf', LogisticRegression(**params))])
    elif model_name == 'rf':
        params = {'n_estimators': trial.suggest_int('rf_n_estimators', 100, 1000),
                  'max_depth': trial.suggest_int('rf_max_depth', 1, 15),
                  'min_samples_leaf': trial.suggest_int('rf_min_samples_leaf', 1, 30),
                  'max_features': trial.suggest_float('rf_max_features', 0.2, 1.0),
                  'min_samples_split': trial.suggest_int('rf_min_samples_split', 2, 30),
                  'random_state': 42,
                  'max_samples': trial.suggest_float('rf_max_samples', 0.4, 1.0),
                  'class_weight': trial.suggest_categorical('rf_class_weight', [None, 'balanced']),
                  'bootstrap': True,
                  'n_jobs': -1,}
        model = RandomForestClassifier(**params)
    elif model_name == 'xgb':
        params = {'n_estimators': 1000,
                  'max_depth': trial.suggest_int('xgb_max_depth', 2, 10),
                  'learning_rate': trial.suggest_float('xgb_learning_rate', 0.005, 1.0, log=True),
                  'subsample': trial.suggest_float('xgb_subsample', 0.65, 1.0),
                  'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.6, 1.0),
                  'reg_alpha': trial.suggest_float('xgb_reg_alpha', 0.001, 4.0, log=True),
                  'reg_lambda': trial.suggest_float('xgb_reg_lambda', 0.001, 4.0, log=True),
                  'eval_metric': 'auc',
                  'tree_method': 'hist',
                  'random_state': 42,
                  'n_jobs': -1,
                  'early_stopping_rounds': 50,
                  'max_bin': trial.suggest_int('xgb_max_bins', 64, 512),
                  'min_child_weight': trial.suggest_int('xgb_min_child_weight', 1, 20, log=True),
                  'objective': 'binary:logistic',}
        model = XGBClassifier(**params)
    elif model_name == 'lgb':
        params = {'n_estimators': 1000,
                  'learning_rate': trial.suggest_float('lgb_learning_rate', 0.005, 1.0, log=True),
                  'num_leaves': trial.suggest_int('lgb_num_leaves', 8, 256, log=True),
                  'min_child_samples': trial.suggest_int('lgb_min_child_samples', 8, 200, log=True),
                  'subsample': trial.suggest_float('lgb_subsample', 0.6, 1.0),
                  'subsample_freq': 1,
                  'colsample_bytree': trial.suggest_float('lgb_colsample_bytree', 0.6, 1.0),
                  'reg_alpha': trial.suggest_float('lgb_reg_alpha', 0.001, 4.0, log=True),
                  'reg_lambda': trial.suggest_float('lgb_reg_lambda', 0.001, 4.0, log=True),
                  'n_jobs': -1,
                  'metric': ['auc', 'binary_logloss'],
                  'random_state': 42,
                  'verbosity': -1,
                  'max_bin': trial.suggest_int('lgb_max_bins', 64, 512),
                  'path_smooth': trial.suggest_float('lgb_path_smooth', 0.0, 10.0),}
        model = LGBMClassifier(**params)
    elif model_name == 'cat':
        params = {'n_estimators': 1000,
                  'learning_rate': trial.suggest_float('cat_learning_rate', 0.005, 1.0, log=True),
                  'depth': trial.suggest_int('cat_depth', 2, 10),
                  'l2_leaf_reg': trial.suggest_float('cat_l2_leaf_reg', 0.01, 10.0, log=True),
                  'random_seed': 42,
                  'random_strength': trial.suggest_float('cat_random_strength', 0.001, 10.0, log=True),
                  'subsample': trial.suggest_float('cat_subsample', 0.6, 1.0),
                  'bootstrap_type': 'MVS',
                  'eval_metric': 'AUC',
                  'loss_function': 'Logloss',
                  'verbose': False,
                  'border_count': trial.suggest_int('cat_border_count', 32, 152),
                  'auto_class_weights': trial.suggest_categorical('cat_auto_class_weights', [None, 'Balanced']),
                'thread_count': -1,
                  'custom_metric': ['Accuracy']}
        model = CatBoostClassifier(**params)
    return model

In [231]:
def fit_model(model, name, x_tr, y_tr, x_val, y_val):
    if name == 'xgb':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], verbose=False)
        return model.best_iteration + 1
    if name == 'lgb':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False), lgb.log_evaluation(0)],)
        return model.best_iteration_
    if name == 'cat':
        model.fit(x_tr, y_tr, eval_set=(x_val, y_val), verbose=False, early_stopping_rounds=50,)
        return model.get_best_iteration() + 1
    model.fit(x_tr, y_tr)
    return None

In [232]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
def make_objective(name):
    def objective(trial):
        scores, iters = [], []
        for tr_idx, val_idx in skf.split(x_tmp, y_tmp):
            xa, xb = x_tmp.iloc[tr_idx], x_tmp.iloc[val_idx]
            ya, yb = y_tmp.iloc[tr_idx], y_tmp.iloc[val_idx]

            p = clone(prep)
            xa_t = p.fit_transform(xa, ya)
            xb_t = p.transform(xb)

            model = unic(trial, name)
            n_trees = fit_model(model, name, xa_t, ya, xb_t, yb)
            proba = model.predict_proba(xb_t)[:, 1]
            scores.append(roc_auc_score(yb, proba))
            if n_trees is not None:
                iters.append(n_trees)
        if iters:
            trial.set_user_attr("best_iter", int(np.mean(iters)))
        trial.set_user_attr("auc_std", float(np.std(scores)))
        return float(np.mean(scores))
    return objective


In [233]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

n_trials = {'cat': 13, 'lgb': 18, 'lr': 13, 'xgb': 18, 'rf': 14}
studies = {}
for name, n in n_trials.items():
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=5),
        study_name=name,
    )
    study.optimize(make_objective(name), n_trials=n)
    studies[name] = study
    t = study.best_trial
    print(f'{name}: auc = {study.best_value:.5f} +- {t.user_attrs['auc_std']:.5f}')

cat: auc = 0.92647 +- 0.00142
lgb: auc = 0.92686 +- 0.00137
lr: auc = 0.90119 +- 0.00185
xgb: auc = 0.92811 +- 0.00146
rf: auc = 0.91737 +- 0.00147


In [234]:
def final_model(name, study):
    best = study.best_trial
    model = unic(FixedTrial(best.params), name)
    n = best.user_attrs.get("best_iter")
    if n is not None:
        n_final = int(n * 1.1)
        if name == 'xgb':
            model.set_params(n_estimators=n_final, early_stopping_rounds=None)
        elif name == 'lgb':
            model.set_params(n_estimators=n_final)
        elif name == 'cat':
            model.set_params(n_estimators=n_final)
    return model

stopping который передается в fit нужно не отключать

In [238]:
import math

def relative_position(value, dist):
    low, high = dist.low, dist.high
    if dist.log:
        return (math.log(value) - math.log(low)) / (math.log(high) - math.log(low))
    return (value - low) / (high - low)

def boundary_report(study, top_k=5, margin=0.1):
    top = sorted(study.trials, key=lambda t: t.value, reverse=True)[:top_k]
    for name, dist in study.best_trial.distributions.items():
        if not hasattr(dist, "low"):
            continue
        positions = [relative_position(t.params[name], dist) for t in top]
        mean_pos = sum(positions) / len(positions)
        flag = "<- у границы" if mean_pos < margin or mean_pos > 1 - margin else ""
        print(f"{name:28s} {mean_pos:.2f} {flag}")

In [ ]:
for name, study in studies.items():
    print(f"\n {name}")
    boundary_report(study)

Проверка, какие параметры близки к порогу

In [240]:
cv_oof = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

def get_oof(name, study, cv):
    oof = np.full(len(x_tmp), np.nan)

    for tr_idx, val_idx in cv.split(x_tmp, y_tmp):
        xa, xb = x_tmp.iloc[tr_idx], x_tmp.iloc[val_idx]
        ya = y_tmp.iloc[tr_idx]

        prep_fold = clone(prep)
        xa_t = prep_fold.fit_transform(xa, ya)
        xb_t = prep_fold.transform(xb)

        model = final_model(name, study)
        model.fit(xa_t, ya)
        oof[val_idx] = model.predict_proba(xb_t)[:, 1]

    prep_full = clone(prep)
    x_tmp_t = prep_full.fit_transform(x_tmp, y_tmp)
    x_test_t = prep_full.transform(x_test)

    model = final_model(name, study)
    model.fit(x_tmp_t, y_tmp)
    test_pred = model.predict_proba(x_test_t)[:, 1]
    return oof, test_pred


oof_df = pd.DataFrame(index=x_tmp.index)
test_df = pd.DataFrame(index=x_test.index)

for name, study in studies.items():
    oof_df[name], test_df[name] = get_oof(name, study, cv_oof)

learn oof

In [246]:
print("nan:", oof_df.isna().sum().sum(), "| формы:", oof_df.shape, test_df.shape)
print("доля положительных в y_tmp:", round(y_tmp.mean(), 3), "\n")

for name in oof_df:
    print(
        f"{name:4s}"
        f" oof_auc={roc_auc_score(y_tmp, oof_df[name]):.5f}"
        f" best_value={studies[name].best_value:.5f}"
        f" test_auc={roc_auc_score(y_test, test_df[name]):.5f}"
        f" mean_pred={oof_df[name].mean():.3f}"
    )


nan: 0 | формы: (26029, 5) (6508, 5)
доля положительных в y_tmp: 0.241 

cat  oof_auc=0.92611 best_value=0.92647 test_auc=0.92495 mean_pred=0.353
lgb  oof_auc=0.92683 best_value=0.92686 test_auc=0.92634 mean_pred=0.241
lr   oof_auc=0.90101 best_value=0.90119 test_auc=0.90152 mean_pred=0.241
xgb  oof_auc=0.92774 best_value=0.92811 test_auc=0.92671 mean_pred=0.242
rf   oof_auc=0.91696 best_value=0.91737 test_auc=0.91684 mean_pred=0.353


In [247]:
print("xgb один:", roc_auc_score(y_tmp, oof_df['xgb']))
print("среднее 3 бустинга:", roc_auc_score(y_tmp, oof_df[['xgb', 'lgb', 'cat']].mean(axis=1)))
print("среднее всех 5:", roc_auc_score(y_tmp, oof_df.mean(axis=1)))

xgb один: 0.9277408734670537
среднее 3 бустинга: 0.9276966249359234
среднее всех 5: 0.9247434380174888


In [243]:

print(oof_df.corr(method='spearman').round(3))
resid = pd.DataFrame({c: y_tmp - oof_df[c] for c in oof_df})
print(resid.corr().round(3))
ranks = oof_df.rank(pct=True)
print("среднее рангов всех 5:", roc_auc_score(y_tmp, ranks.mean(axis=1)))

       cat    lgb     lr    xgb     rf
cat  1.000  0.989  0.933  0.990  0.962
lgb  0.989  1.000  0.931  0.996  0.971
lr   0.933  0.931  1.000  0.929  0.934
xgb  0.990  0.996  0.929  1.000  0.972
rf   0.962  0.971  0.934  0.972  1.000
       cat    lgb     lr    xgb     rf
cat  1.000  0.937  0.875  0.940  0.965
lgb  0.937  1.000  0.923  0.996  0.922
lr   0.875  0.923  1.000  0.926  0.893
xgb  0.940  0.996  0.926  1.000  0.925
rf   0.965  0.922  0.893  0.925  1.000
среднее рангов всех 5: 0.9250303331449108


на сколько модели похожи, на сколько у них одинаковые ошибки

In [249]:
from itertools import combinations

ranks = oof_df.rank(pct=True)
results = []
for k in range(1, 6):
    for subset in combinations(oof_df.columns, k):
        auc = roc_auc_score(y_tmp, ranks[list(subset)].mean(axis=1))
        results.append((auc, subset))

for auc, subset in sorted(results, reverse=True)[:8]:
    print(f"{auc:.5f}", subset)

0.92774 ('xgb',)
0.92769 ('cat', 'lgb', 'xgb')
0.92762 ('lgb', 'xgb')
0.92761 ('cat', 'xgb')
0.92723 ('cat', 'lgb')
0.92683 ('lgb',)
0.92650 ('cat', 'lgb', 'xgb', 'rf')
0.92611 ('cat',)


auc по парам, показывает есть ли смысл вообще делать ансамбль если одна модель лучше справляется чем по парам. И в целом показывает какие пары и соло лучше всего справляются